In [2]:

# ============================================================
# CELL 1 — Imports
# ============================================================
import os
import re
import subprocess
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

print("✅ Imports done")


✅ Imports done


In [3]:

# ============================================================
# CELL 2 — Paths and thresholds
# ============================================================
BASE_DIR      = "/Users/abey/Documents/pause_alignment"   # ← change this
REFERENCE_DIR = os.path.join(BASE_DIR, "reference")
MODELS_DIR    = os.path.join(BASE_DIR, "models")

FFMPEG_BIN = "ffmpeg"  # change to full path if not in PATH

# silence detection parameters
SILENCE_DB           = -40    # dB threshold
MIN_SILENCE_DURATION = 0.5    # minimum pause duration in seconds

# matching — hard limit beyond which a pair is never assigned
POSITION_HARD_LIMIT = 3.0     # seconds — pairs beyond this get infinite cost

# weighted cost parameters — calibrate with editor later
POSITION_WEIGHT = 0.3         # weight for position component of cost
DURATION_WEIGHT = 0.7         # weight for duration component of cost
POSITION_SCALE  = 3.0         # seconds — position diff of this size = cost 1.0

# pass/fail thresholds — calibrate with editor later
PAUSE_COUNT_THRESHOLD     = 2     # max acceptable pause count difference
POSITION_OFFSET_THRESHOLD = 0.5   # max acceptable median position offset in seconds
DURATION_RATIO_MIN        = 0.75  # TTS pause at least 75% as long as reference
DURATION_RATIO_MAX        = 1.25  # TTS pause at most 125% as long as reference

# degraded — reference too dense to be a reliable alignment signal
REF_PAUSES_PER_SECOND_LIMIT = 1.0  # pauses per second above this = degraded

print(f"Silence DB                : {SILENCE_DB} dB")
print(f"Min Silence Duration      : {MIN_SILENCE_DURATION}s")
print(f"Position Hard Limit       : {POSITION_HARD_LIMIT}s")
print(f"Position / Duration Weight: {POSITION_WEIGHT} / {DURATION_WEIGHT}")
print(f"Position Scale            : {POSITION_SCALE}s")
print(f"Pause Count Threshold     : ±{PAUSE_COUNT_THRESHOLD}")
print(f"Position Offset Threshold : {POSITION_OFFSET_THRESHOLD}s")
print(f"Duration Ratio Range      : {DURATION_RATIO_MIN} – {DURATION_RATIO_MAX}")
print(f"Degraded Pauses/sec limit : {REF_PAUSES_PER_SECOND_LIMIT}")
print("✅ Paths and thresholds set")



Silence DB                : -40 dB
Min Silence Duration      : 0.5s
Position Hard Limit       : 3.0s
Position / Duration Weight: 0.3 / 0.7
Position Scale            : 3.0s
Pause Count Threshold     : ±2
Position Offset Threshold : 0.5s
Duration Ratio Range      : 0.75 – 1.25
Degraded Pauses/sec limit : 1.0
✅ Paths and thresholds set


In [4]:

# ============================================================
# CELL 3 — get_pauses function
# ============================================================
def get_pauses(audio_path):
    """
    Uses ffmpeg silencedetect to find pause intervals.
    Returns list of dicts with start, end, duration per pause.
    Also returns audio duration for degraded check.
    """
    cmd = [
        FFMPEG_BIN, "-y",
        "-i", audio_path,
        "-af", f"silencedetect=noise={SILENCE_DB}dB:d={MIN_SILENCE_DURATION}",
        "-f", "null", "-"
    ]

    try:
        res    = subprocess.run(cmd, stderr=subprocess.PIPE, text=True, timeout=30)
        output = res.stderr
    except subprocess.TimeoutExpired:
        print(f"  ⚠️  ffmpeg timeout: {os.path.basename(audio_path)}")
        return [], 0.0
    except Exception as e:
        print(f"  ⚠️  ffmpeg error: {e}")
        return [], 0.0

    # extract audio duration from ffmpeg output
    audio_duration = 0.0
    dur_match = re.search(r"Duration: (\d+):(\d+):([\d\.]+)", output)
    if dur_match:
        h, m, s      = dur_match.groups()
        audio_duration = int(h) * 3600 + int(m) * 60 + float(s)

    starts, ends = [], []
    for line in output.split('\n'):
        if "silence_start" in line:
            m = re.search(r"silence_start: ([\d\.]+)", line)
            if m:
                starts.append(float(m.group(1)))
        elif "silence_end" in line:
            m = re.search(r"silence_end: ([\d\.]+)", line)
            if m:
                ends.append(float(m.group(1)))

    # handle case where audio starts with silence
    if len(ends) > len(starts):
        ends = ends[1:]

    pauses = []
    for s, e in zip(starts, ends):
        pauses.append({
            "start"   : round(s, 3),
            "end"     : round(e, 3),
            "duration": round(e - s, 3)
        })

    return pauses, audio_duration

print("✅ get_pauses defined")




✅ get_pauses defined


In [5]:

# ============================================================
# CELL 4 — compare_pauses function (Hungarian + weighted cost)
# ============================================================
def compare_pauses(ref_pauses, tts_pauses):
    """
    Matches ref pauses to TTS pauses using Hungarian algorithm
    with weighted cost combining position and duration.

    Cost between ref pause R and TTS pause T:
        position_cost = position_diff / POSITION_SCALE
        duration_cost = duration_diff / ref_duration
        cost = POSITION_WEIGHT * position_cost + DURATION_WEIGHT * duration_cost

    Pairs where position diff exceeds POSITION_HARD_LIMIT get infinite cost
    and are never assigned. Unassigned pauses on either side contribute to
    count delta but not to position/duration metrics.
    """
    ref_count = len(ref_pauses)
    tts_count = len(tts_pauses)

    # if either has no pauses — count delta only
    if ref_count == 0 or tts_count == 0:
        return {
            "ref_count"          : ref_count,
            "tts_count"          : tts_count,
            "matched_count"      : 0,
            "unmatched_ref"      : ref_count,
            "unmatched_tts"      : tts_count,
            "count_delta"        : abs(ref_count - tts_count),
            "med_position_offset": None,
            "med_duration_ratio" : None,
            "count_pass"         : abs(ref_count - tts_count) <= PAUSE_COUNT_THRESHOLD,
            "position_pass"      : None,
            "duration_pass"      : None,
        }

    # ── build cost matrix ──
    INF = 1e9
    cost_matrix = np.full((ref_count, tts_count), INF)

    for i, ref_p in enumerate(ref_pauses):
        ref_mid = (ref_p["start"] + ref_p["end"]) / 2
        for j, tts_p in enumerate(tts_pauses):
            tts_mid      = (tts_p["start"] + tts_p["end"]) / 2
            position_diff = abs(ref_mid - tts_mid)

            if position_diff > POSITION_HARD_LIMIT:
                continue  # leave as INF — never assigned

            duration_diff  = abs(ref_p["duration"] - tts_p["duration"])
            position_cost  = position_diff / POSITION_SCALE
            duration_cost  = duration_diff / ref_p["duration"] if ref_p["duration"] > 0 else 0.0
            cost_matrix[i, j] = (
                POSITION_WEIGHT * position_cost +
                DURATION_WEIGHT * duration_cost
            )

    # ── run Hungarian ──
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    # ── collect valid assignments — reject infinite cost pairs ──
    position_offsets = []
    duration_ratios  = []
    matched_ref      = set()
    matched_tts      = set()

    for r, c in zip(row_ind, col_ind):
        if cost_matrix[r, c] >= INF:
            continue  # positionally impossible — treat as unmatched

        ref_p = ref_pauses[r]
        tts_p = tts_pauses[c]
        ref_mid = (ref_p["start"] + ref_p["end"]) / 2
        tts_mid = (tts_p["start"] + tts_p["end"]) / 2

        position_offsets.append(abs(ref_mid - tts_mid))
        if ref_p["duration"] > 0:
            duration_ratios.append(tts_p["duration"] / ref_p["duration"])

        matched_ref.add(r)
        matched_tts.add(c)

    matched_count  = len(matched_ref)
    unmatched_ref  = ref_count - matched_count
    unmatched_tts  = tts_count - len(matched_tts)
    count_delta    = unmatched_ref + unmatched_tts

    med_position_offset = round(float(np.median(position_offsets)), 3) if position_offsets else None
    med_duration_ratio  = round(float(np.median(duration_ratios)),  3) if duration_ratios  else None

    count_pass    = count_delta <= PAUSE_COUNT_THRESHOLD
    position_pass = (med_position_offset <= POSITION_OFFSET_THRESHOLD) if med_position_offset is not None else None
    duration_pass = (DURATION_RATIO_MIN <= med_duration_ratio <= DURATION_RATIO_MAX) if med_duration_ratio is not None else None

    return {
        "ref_count"          : ref_count,
        "tts_count"          : tts_count,
        "matched_count"      : matched_count,
        "unmatched_ref"      : unmatched_ref,
        "unmatched_tts"      : unmatched_tts,
        "count_delta"        : count_delta,
        "med_position_offset": med_position_offset,
        "med_duration_ratio" : med_duration_ratio,
        "count_pass"         : count_pass,
        "position_pass"      : position_pass,
        "duration_pass"      : duration_pass,
    }

print("✅ compare_pauses defined")




✅ compare_pauses defined


In [6]:

# ============================================================
# CELL 5 — Startup validation
# ============================================================
try:
    subprocess.run([FFMPEG_BIN, "-version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print("✅ ffmpeg found")
except FileNotFoundError:
    raise FileNotFoundError(
        f"ffmpeg not found at '{FFMPEG_BIN}'\n"
        f"Install: brew install ffmpeg"
    )

for folder in [BASE_DIR, REFERENCE_DIR, MODELS_DIR]:
    if not os.path.exists(folder):
        raise FileNotFoundError(f"Folder not found: {folder}")
print("✅ Top level folders found")

model_folders = sorted([
    d for d in os.listdir(MODELS_DIR)
    if os.path.isdir(os.path.join(MODELS_DIR, d))
])
if not model_folders:
    raise ValueError(f"No model folders found in {MODELS_DIR}")
print(f"✅ Models found: {model_folders}")

model_samples = {}
for model in model_folders:
    model_path = os.path.join(MODELS_DIR, model)
    wav_files  = sorted([f for f in os.listdir(model_path) if f.endswith(".wav")])
    model_samples[model] = wav_files
    print(f"   {model}: {len(wav_files)} samples")

reference_filenames = set(model_samples[model_folders[0]])
for model in model_folders[1:]:
    current_filenames = set(model_samples[model])
    if current_filenames != reference_filenames:
        missing = reference_filenames - current_filenames
        extra   = current_filenames - reference_filenames
        raise ValueError(
            f"Model '{model}' has mismatched filenames.\n"
            f"  Missing : {missing}\n"
            f"  Extra   : {extra}"
        )
print("✅ All models have identical filenames")

sample_names = model_samples[model_folders[0]]
for wav_file in sample_names:
    ref_path = os.path.join(REFERENCE_DIR, wav_file)
    if not os.path.exists(ref_path):
        raise FileNotFoundError(f"Missing reference for {wav_file} — expected: {ref_path}")
print("✅ All reference files found")

total = len(model_folders) * len(sample_names)
print(f"\nReady: {len(model_folders)} models × {len(sample_names)} samples = {total} evaluations")




✅ ffmpeg found
✅ Top level folders found
✅ Models found: ['m1']
   m1: 2 samples
✅ All models have identical filenames
✅ All reference files found

Ready: 1 models × 2 samples = 2 evaluations


In [7]:
# ============================================================
# CELL 1 — Imports
# ============================================================
import os
import re
import subprocess
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

print("✅ Imports done")


# ============================================================
# CELL 2 — Paths and thresholds
# ============================================================
BASE_DIR      = "/Users/abey/Documents/pause_alignment"   # ← change this
REFERENCE_DIR = os.path.join(BASE_DIR, "reference")
MODELS_DIR    = os.path.join(BASE_DIR, "models")

FFMPEG_BIN = "ffmpeg"  # change to full path if not in PATH

# silence detection parameters
SILENCE_DB           = -40    # dB threshold
MIN_SILENCE_DURATION = 0.5    # minimum pause duration in seconds

# matching — hard limit beyond which a pair is never assigned
POSITION_HARD_LIMIT = 3.0     # seconds — pairs beyond this get infinite cost

# weighted cost parameters — calibrate with editor later
POSITION_WEIGHT = 0.3         # weight for position component of cost
DURATION_WEIGHT = 0.7         # weight for duration component of cost
POSITION_SCALE  = 3.0         # seconds — position diff of this size = cost 1.0

# pass/fail thresholds — calibrate with editor later
PAUSE_COUNT_THRESHOLD     = 20     # max acceptable pause count difference
POSITION_OFFSET_THRESHOLD = 0.5   # max acceptable median position offset in seconds
DURATION_RATIO_MIN        = 0.75  # TTS pause at least 75% as long as reference
DURATION_RATIO_MAX        = 1.25  # TTS pause at most 125% as long as reference

# degraded — reference too dense to be a reliable alignment signal
REF_PAUSES_PER_SECOND_LIMIT = 1.0  # pauses per second above this = degraded

print(f"Silence DB                : {SILENCE_DB} dB")
print(f"Min Silence Duration      : {MIN_SILENCE_DURATION}s")
print(f"Position Hard Limit       : {POSITION_HARD_LIMIT}s")
print(f"Position / Duration Weight: {POSITION_WEIGHT} / {DURATION_WEIGHT}")
print(f"Position Scale            : {POSITION_SCALE}s")
print(f"Pause Count Threshold     : ±{PAUSE_COUNT_THRESHOLD}")
print(f"Position Offset Threshold : {POSITION_OFFSET_THRESHOLD}s")
print(f"Duration Ratio Range      : {DURATION_RATIO_MIN} – {DURATION_RATIO_MAX}")
print(f"Degraded Pauses/sec limit : {REF_PAUSES_PER_SECOND_LIMIT}")
print("✅ Paths and thresholds set")


# ============================================================
# CELL 3 — get_pauses function
# ============================================================
def get_pauses(audio_path):
    """
    Uses ffmpeg silencedetect to find pause intervals.
    Returns list of dicts with start, end, duration per pause.
    Also returns audio duration for degraded check.
    """
    cmd = [
        FFMPEG_BIN, "-y",
        "-i", audio_path,
        "-af", f"silencedetect=noise={SILENCE_DB}dB:d={MIN_SILENCE_DURATION}",
        "-f", "null", "-"
    ]

    try:
        res    = subprocess.run(cmd, stderr=subprocess.PIPE, text=True, timeout=30)
        output = res.stderr
    except subprocess.TimeoutExpired:
        print(f"  ⚠️  ffmpeg timeout: {os.path.basename(audio_path)}")
        return [], 0.0
    except Exception as e:
        print(f"  ⚠️  ffmpeg error: {e}")
        return [], 0.0

    # extract audio duration from ffmpeg output
    audio_duration = 0.0
    dur_match = re.search(r"Duration: (\d+):(\d+):([\d\.]+)", output)
    if dur_match:
        h, m, s      = dur_match.groups()
        audio_duration = int(h) * 3600 + int(m) * 60 + float(s)

    starts, ends = [], []
    for line in output.split('\n'):
        if "silence_start" in line:
            m = re.search(r"silence_start: ([\d\.]+)", line)
            if m:
                starts.append(float(m.group(1)))
        elif "silence_end" in line:
            m = re.search(r"silence_end: ([\d\.]+)", line)
            if m:
                ends.append(float(m.group(1)))

    # handle case where audio starts with silence
    if len(ends) > len(starts):
        ends = ends[1:]

    pauses = []
    for s, e in zip(starts, ends):
        pauses.append({
            "start"   : round(s, 3),
            "end"     : round(e, 3),
            "duration": round(e - s, 3)
        })

    return pauses, audio_duration

print("✅ get_pauses defined")


# ============================================================
# CELL 4 — compare_pauses function (Hungarian + weighted cost)
# ============================================================
def compare_pauses(ref_pauses, tts_pauses):
    """
    Matches ref pauses to TTS pauses using Hungarian algorithm
    with weighted cost combining position and duration.

    Cost between ref pause R and TTS pause T:
        position_cost = position_diff / POSITION_SCALE
        duration_cost = duration_diff / ref_duration
        cost = POSITION_WEIGHT * position_cost + DURATION_WEIGHT * duration_cost

    Pairs where position diff exceeds POSITION_HARD_LIMIT get infinite cost
    and are never assigned. Unassigned pauses on either side contribute to
    count delta but not to position/duration metrics.
    """
    ref_count = len(ref_pauses)
    tts_count = len(tts_pauses)

    # if either has no pauses — count delta only
    if ref_count == 0 or tts_count == 0:
        return {
            "ref_count"          : ref_count,
            "tts_count"          : tts_count,
            "matched_count"      : 0,
            "unmatched_ref"      : ref_count,
            "unmatched_tts"      : tts_count,
            "count_delta"        : abs(ref_count - tts_count),
            "med_position_offset": None,
            "med_duration_ratio" : None,
            "count_pass"         : abs(ref_count - tts_count) <= PAUSE_COUNT_THRESHOLD,
            "position_pass"      : None,
            "duration_pass"      : None,
        }

    # ── build cost matrix ──
    INF = 1e9
    cost_matrix = np.full((ref_count, tts_count), INF)

    for i, ref_p in enumerate(ref_pauses):
        ref_mid = (ref_p["start"] + ref_p["end"]) / 2
        for j, tts_p in enumerate(tts_pauses):
            tts_mid      = (tts_p["start"] + tts_p["end"]) / 2
            position_diff = abs(ref_mid - tts_mid)

            if position_diff > POSITION_HARD_LIMIT:
                continue  # leave as INF — never assigned

            duration_diff  = abs(ref_p["duration"] - tts_p["duration"])
            position_cost  = position_diff / POSITION_SCALE
            duration_cost  = duration_diff / ref_p["duration"] if ref_p["duration"] > 0 else 0.0
            cost_matrix[i, j] = (
                POSITION_WEIGHT * position_cost +
                DURATION_WEIGHT * duration_cost
            )

    # ── run Hungarian ──
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    # ── collect valid assignments — reject infinite cost pairs ──
    position_offsets = []
    duration_ratios  = []
    matched_ref      = set()
    matched_tts      = set()

    for r, c in zip(row_ind, col_ind):
        if cost_matrix[r, c] >= INF:
            continue  # positionally impossible — treat as unmatched

        ref_p = ref_pauses[r]
        tts_p = tts_pauses[c]
        ref_mid = (ref_p["start"] + ref_p["end"]) / 2
        tts_mid = (tts_p["start"] + tts_p["end"]) / 2

        position_offsets.append(abs(ref_mid - tts_mid))
        if ref_p["duration"] > 0:
            duration_ratios.append(tts_p["duration"] / ref_p["duration"])

        matched_ref.add(r)
        matched_tts.add(c)

    matched_count  = len(matched_ref)
    unmatched_ref  = ref_count - matched_count
    unmatched_tts  = tts_count - len(matched_tts)
    count_delta    = unmatched_ref + unmatched_tts

    med_position_offset = round(float(np.median(position_offsets)), 3) if position_offsets else None
    med_duration_ratio  = round(float(np.median(duration_ratios)),  3) if duration_ratios  else None

    count_pass    = count_delta <= PAUSE_COUNT_THRESHOLD
    position_pass = (med_position_offset <= POSITION_OFFSET_THRESHOLD) if med_position_offset is not None else None
    duration_pass = (DURATION_RATIO_MIN <= med_duration_ratio <= DURATION_RATIO_MAX) if med_duration_ratio is not None else None

    return {
        "ref_count"          : ref_count,
        "tts_count"          : tts_count,
        "matched_count"      : matched_count,
        "unmatched_ref"      : unmatched_ref,
        "unmatched_tts"      : unmatched_tts,
        "count_delta"        : count_delta,
        "med_position_offset": med_position_offset,
        "med_duration_ratio" : med_duration_ratio,
        "count_pass"         : count_pass,
        "position_pass"      : position_pass,
        "duration_pass"      : duration_pass,
    }

print("✅ compare_pauses defined")


# ============================================================
# CELL 5 — Startup validation
# ============================================================
try:
    subprocess.run([FFMPEG_BIN, "-version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print("✅ ffmpeg found")
except FileNotFoundError:
    raise FileNotFoundError(
        f"ffmpeg not found at '{FFMPEG_BIN}'\n"
        f"Install: brew install ffmpeg"
    )

for folder in [BASE_DIR, REFERENCE_DIR, MODELS_DIR]:
    if not os.path.exists(folder):
        raise FileNotFoundError(f"Folder not found: {folder}")
print("✅ Top level folders found")

model_folders = sorted([
    d for d in os.listdir(MODELS_DIR)
    if os.path.isdir(os.path.join(MODELS_DIR, d))
])
if not model_folders:
    raise ValueError(f"No model folders found in {MODELS_DIR}")
print(f"✅ Models found: {model_folders}")

model_samples = {}
for model in model_folders:
    model_path = os.path.join(MODELS_DIR, model)
    wav_files  = sorted([f for f in os.listdir(model_path) if f.endswith(".wav")])
    model_samples[model] = wav_files
    print(f"   {model}: {len(wav_files)} samples")

reference_filenames = set(model_samples[model_folders[0]])
for model in model_folders[1:]:
    current_filenames = set(model_samples[model])
    if current_filenames != reference_filenames:
        missing = reference_filenames - current_filenames
        extra   = current_filenames - reference_filenames
        raise ValueError(
            f"Model '{model}' has mismatched filenames.\n"
            f"  Missing : {missing}\n"
            f"  Extra   : {extra}"
        )
print("✅ All models have identical filenames")

sample_names = model_samples[model_folders[0]]
for wav_file in sample_names:
    ref_path = os.path.join(REFERENCE_DIR, wav_file)
    if not os.path.exists(ref_path):
        raise FileNotFoundError(f"Missing reference for {wav_file} — expected: {ref_path}")
print("✅ All reference files found")

total = len(model_folders) * len(sample_names)
print(f"\nReady: {len(model_folders)} models × {len(sample_names)} samples = {total} evaluations")


# ============================================================
# CELL 6 — Main evaluation loop
# ============================================================
results = []

for model in model_folders:
    print(f"\n{'='*50}")
    print(f"Model: {model}")
    print(f"{'='*50}")

    for wav_file in model_samples[model]:
        sample_name = os.path.splitext(wav_file)[0]
        tts_path    = os.path.join(MODELS_DIR, model, wav_file)
        ref_path    = os.path.join(REFERENCE_DIR, wav_file)

        print(f"\n  Sample : {sample_name}")

        try:
            ref_pauses, ref_duration = get_pauses(ref_path)
            tts_pauses, _            = get_pauses(tts_path)

            print(f"  Ref    : {len(ref_pauses)} pauses | duration {round(ref_duration, 2)}s")
            print(f"  TTS    : {len(tts_pauses)} pauses")

            # ── degraded check — ref too dense ──
            ref_pauses_per_sec = len(ref_pauses) / ref_duration if ref_duration > 0 else 0
            is_degraded        = ref_pauses_per_sec > REF_PAUSES_PER_SECOND_LIMIT
            ref_flag           = "REF_DENSE" if is_degraded else "—"

            if is_degraded:
                print(f"  ⚠️  Ref pauses/sec={round(ref_pauses_per_sec, 2)} — degraded")

            # ── compare pauses ──
            cmp = compare_pauses(ref_pauses, tts_pauses)

            # ── final pass/fail ──
            failures = []
            if not cmp["count_pass"]:
                failures.append("Count")
            if cmp["position_pass"] is False:
                failures.append("Position")
            if cmp["duration_pass"] is False:
                failures.append("Duration")

            final_pass = "✅ PASS" if not failures else f"❌ FAIL ({', '.join(failures)})"

            print(f"  Matched: {cmp['matched_count']} | Unmatched ref: {cmp['unmatched_ref']} | Unmatched tts: {cmp['unmatched_tts']}")
            print(f"  Result : {final_pass} | Count Δ: {cmp['count_delta']} | Pos: {cmp['med_position_offset']}s | Dur ratio: {cmp['med_duration_ratio']}")

            results.append({
                "Model"            : model,
                "Sample"           : sample_name,
                "Ref Pauses"       : cmp["ref_count"],
                "TTS Pauses"       : cmp["tts_count"],
                "Matched"          : cmp["matched_count"],
                "Unmatched Ref"    : cmp["unmatched_ref"],
                "Unmatched TTS"    : cmp["unmatched_tts"],
                "Count Delta"      : cmp["count_delta"],
                "Med Pos Offset"   : cmp["med_position_offset"],
                "Med Dur Ratio"    : cmp["med_duration_ratio"],
                "Count Pass"       : "✅" if cmp["count_pass"] else "❌",
                "Position Pass"    : "✅" if cmp["position_pass"] else "❌" if cmp["position_pass"] is not None else "—",
                "Duration Pass"    : "✅" if cmp["duration_pass"] else "❌" if cmp["duration_pass"] is not None else "—",
                "Final Pass"       : final_pass,
                "Ref Flag"         : ref_flag,
                "_is_degraded"     : is_degraded,
            })

        except Exception as e:
            print(f"  🚨 ERROR: {e}")
            results.append({
                "Model"            : model,
                "Sample"           : sample_name,
                "Ref Pauses"       : None,
                "TTS Pauses"       : None,
                "Matched"          : None,
                "Unmatched Ref"    : None,
                "Unmatched TTS"    : None,
                "Count Delta"      : None,
                "Med Pos Offset"   : None,
                "Med Dur Ratio"    : None,
                "Count Pass"       : "—",
                "Position Pass"    : "—",
                "Duration Pass"    : "—",
                "Final Pass"       : "⚠️ ERROR",
                "Ref Flag"         : "ERROR",
                "_is_degraded"     : False,
            })

print("\n\nAll evaluations complete.")



✅ Imports done
Silence DB                : -40 dB
Min Silence Duration      : 0.5s
Position Hard Limit       : 3.0s
Position / Duration Weight: 0.3 / 0.7
Position Scale            : 3.0s
Pause Count Threshold     : ±20
Position Offset Threshold : 0.5s
Duration Ratio Range      : 0.75 – 1.25
Degraded Pauses/sec limit : 1.0
✅ Paths and thresholds set
✅ get_pauses defined
✅ compare_pauses defined
✅ ffmpeg found
✅ Top level folders found
✅ Models found: ['m1']
   m1: 2 samples
✅ All models have identical filenames
✅ All reference files found

Ready: 1 models × 2 samples = 2 evaluations

Model: m1

  Sample : YASH 2
  Ref    : 99 pauses | duration 332.43s
  TTS    : 76 pauses
  Matched: 55 | Unmatched ref: 44 | Unmatched tts: 21
  Result : ❌ FAIL (Count, Position) | Count Δ: 65 | Pos: 0.799s | Dur ratio: 0.928

  Sample : YASH_01
  Ref    : 76 pauses | duration 254.86s
  TTS    : 99 pauses
  Matched: 55 | Unmatched ref: 21 | Unmatched tts: 44
  Result : ❌ FAIL (Count, Position) | Count Δ: 6

In [8]:

# ============================================================
# CELL 7 — Results and model comparison
# ============================================================
df = pd.DataFrame(results)

# ── Table 1 — full per segment ──
print("\n========== FULL PER-SEGMENT RESULTS ==========")
print(df[[
    "Model", "Sample",
    "Ref Pauses", "TTS Pauses", "Matched",
    "Unmatched Ref", "Unmatched TTS", "Count Delta",
    "Med Pos Offset", "Med Dur Ratio",
    "Count Pass", "Position Pass", "Duration Pass",
    "Final Pass", "Ref Flag"
]].to_string(index=False))

# ── Table 2 — per model summary ──
print("\n========== MODEL COMPARISON SUMMARY ==========")
summary_rows = []

for model in model_folders:
    model_df    = df[df["Model"] == model]
    clean_df    = model_df[~model_df["_is_degraded"] & (model_df["Final Pass"] != "⚠️ ERROR")]
    degraded_df = model_df[model_df["_is_degraded"]]
    total       = len(model_df)

    clean_total = len(clean_df)
    clean_pass  = (clean_df["Final Pass"] == "✅ PASS").sum()

    deg_total   = len(degraded_df)
    deg_pass    = (degraded_df["Final Pass"] == "✅ PASS").sum()

    count_fails    = model_df["Final Pass"].str.contains("Count").sum()
    position_fails = model_df["Final Pass"].str.contains("Position").sum()
    duration_fails = model_df["Final Pass"].str.contains("Duration").sum()
    error_count    = (model_df["Final Pass"] == "⚠️ ERROR").sum()

    valid = model_df[model_df["Med Pos Offset"].notna()]
    med_pos    = round(valid["Med Pos Offset"].median(), 3) if len(valid) > 0 else None
    med_dur    = round(valid["Med Dur Ratio"].median(), 3)  if len(valid) > 0 else None

    summary_rows.append({
        "Model"             : model,
        "Total Segments"    : total,
        "Clean Segments"    : clean_total,
        "Clean Pass Rate"   : f"{clean_pass}/{clean_total}"  if clean_total > 0 else "—",
        "Degraded Segments" : deg_total,
        "Degraded Pass Rate": f"{deg_pass}/{deg_total}"      if deg_total > 0 else "—",
        "Count Fails"       : count_fails,
        "Position Fails"    : position_fails,
        "Duration Fails"    : duration_fails,
        "Errors"            : error_count,
        "Median Pos Offset" : med_pos,
        "Median Dur Ratio"  : med_dur,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# ── Table 3 — model ranking ──
print("\n========== MODEL RANKING ==========")
print("Primary   → Clean Pass Rate (ref pauses/sec <= 1.0 segments only)")
print("Tiebreak1 → Degraded Pass Rate")
print("Tiebreak2 → Median Pos Offset (lowest first)\n")

def parse_rate(rate_str):
    if rate_str == "—":
        return -1
    return int(rate_str.split("/")[0])

summary_df["_clean_pass_num"]    = summary_df["Clean Pass Rate"].apply(parse_rate)
summary_df["_degraded_pass_num"] = summary_df["Degraded Pass Rate"].apply(parse_rate)
summary_df["_med_pos"]           = summary_df["Median Pos Offset"].fillna(999)

ranking = summary_df.sort_values(
    by=["_clean_pass_num", "_degraded_pass_num", "_med_pos"],
    ascending=[False, False, True]
)[[
    "Model", "Clean Pass Rate", "Degraded Pass Rate",
    "Count Fails", "Position Fails", "Duration Fails",
    "Median Pos Offset", "Median Dur Ratio"
]]

print(ranking.to_string(index=False))

print("\n========== WHAT TO LOOK FOR ==========")
print("Clean Pass Rate    → primary ranking — ref pauses/sec <= 1.0 only")
print("Degraded Pass Rate → segments where reference had too many pauses")
print("Count Fails        → TTS has wrong number of pauses — merged or split")
print("Position Fails     → pauses in wrong places — dramatic beats misaligned")
print("Duration Fails     → pauses too short or too long")
print("Unmatched Ref      → ref pauses TTS completely missed")
print("Unmatched TTS      → spurious TTS pauses with no ref counterpart")
print(f"\nThresholds:")
print(f"  Position hard limit : {POSITION_HARD_LIMIT}s")
print(f"  Position weight     : {POSITION_WEIGHT} | Duration weight: {DURATION_WEIGHT}")
print(f"  Position scale      : {POSITION_SCALE}s")
print(f"  Count threshold     : ±{PAUSE_COUNT_THRESHOLD}")
print(f"  Position threshold  : {POSITION_OFFSET_THRESHOLD}s")
print(f"  Duration ratio      : {DURATION_RATIO_MIN} – {DURATION_RATIO_MAX}")



========== FULL PER-SEGMENT RESULTS ==========
Model  Sample  Ref Pauses  TTS Pauses  Matched  Unmatched Ref  Unmatched TTS  Count Delta  Med Pos Offset  Med Dur Ratio Count Pass Position Pass Duration Pass               Final Pass Ref Flag
   m1  YASH 2          99          76       55             44             21           65           0.799          0.928          ❌             ❌             ✅ ❌ FAIL (Count, Position)        —
   m1 YASH_01          76          99       55             21             44           65           0.712          1.068          ❌             ❌             ✅ ❌ FAIL (Count, Position)        —

========== MODEL COMPARISON SUMMARY ==========
Model  Total Segments  Clean Segments Clean Pass Rate  Degraded Segments Degraded Pass Rate  Count Fails  Position Fails  Duration Fails  Errors  Median Pos Offset  Median Dur Ratio
   m1               2               2             0/2                  0                  —            2               2               0    

In [9]:
# final cell in each gate notebook
df.to_csv(os.path.join(BASE_DIR, "results.csv"), index=False)
print("✅ Results saved to results.csv")

✅ Results saved to results.csv
